# Formula AST experiment

Load existing compositional explanation search results as written formula trees, then define each neuron Boolean as its formula in one Z3 solver.

In [ ]:
from pathlib import Path

import z3

from analysis import load_compexp, to_z3

In [ ]:
csv_path = Path("../data/snli_beam_results.csv")
results = load_compexp(csv_path)
results.head()

In [ ]:
formula_rows = results.loc[results["ast"].notna()]
formula_rows[["neuron", "formula", "iou", "ast"]].head()

In [ ]:
def formula_to_string(node):
    """Reconstruct search-format text using only the AST."""
    if node.kind == "atom":
        return node.value
    if node.kind == "constant":
        return str(node.value)
    if node.kind == "NOT":
        return f"(NOT {formula_to_string(node.children[0])})"
    if node.kind in {"AND", "OR"}:
        children = (formula_to_string(child) for child in node.children)
        return "(" + f" {node.kind} ".join(children) + ")"
    raise ValueError(f"Unknown formula node kind {node.kind!r}")


round_trip = formula_rows[["neuron", "formula"]].copy()
round_trip["reconstructed_formula"] = formula_rows["ast"].map(formula_to_string)
round_trip["matches"] = round_trip["formula"] == round_trip["reconstructed_formula"]
mismatches = round_trip.loc[~round_trip["matches"]]
assert mismatches.empty, f"AST round-trip mismatches:\n{mismatches.to_string(index=False)}"
round_trip


## Load solver constraints

Each neuron Boolean equals its formula. Feature names share Boolean variables across neurons; empty and pruned formulas are skipped.

In [ ]:
solver = z3.Solver()
for neuron_id, node in zip(formula_rows["neuron"], formula_rows["ast"]):
    neuron = z3.Bool(f"neuron_{neuron_id}")
    solver.add(neuron == to_z3(node))

len(solver.assertions())